# Calculate alpha density using Matti's quadratic

In [11]:
# Reload Process each time (keep editable install up-to-date)
%load_ext autoreload
%autoreload 2
from IPython.display import clear_output
import numpy as np
from process.main import SingleRun
import process.impurity_radiation as impurity_radiation
from process import physics, data_structure

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Test with real values
Run large tokamak solution file to get realistic values for inputs to Matti's calc.

In [24]:
# Matti's n_alpha calculation
def calc_n_alpha(
    ne,
    t_alpha_confinement,
    fusden_plasma_alpha,
    nd_beam_ions,
    znimp,
    z_he,
    proton_rate_density,
):
    a = ne * t_alpha_confinement * fusden_plasma_alpha
    b = 1 - (nd_beam_ions / ne) - (znimp / ne)
    c = z_he + (proton_rate_density / fusden_plasma_alpha)
    quad = np.polynomial.Polynomial(
        (b**2 / c**2, ((-2 * a * b * c) - 1) / (a * c**2), 1)
    )
    print(f"roots = {quad.roots()}")
    n_alpha_calc = quad.roots()[0] * ne
    print(f"n_alpha = {n_alpha_calc}")


# Values from final eval of LT solution evaluation (ne = 4e19)
z_he = 2
nd_beam_ions = 0.0
proton_rate_density = 7.3151716401275e13
fusden_plasma_alpha = 2.8780901503218544e16
t_alpha_confinement = 13.286364626812315
znimp = 1.38e18
ne = 7e19

calc_n_alpha(
    ne=ne,
    t_alpha_confinement=t_alpha_confinement,
    fusden_plasma_alpha=fusden_plasma_alpha,
    nd_beam_ions=nd_beam_ions,
    znimp=znimp,
    z_he=z_he,
    proton_rate_density=proton_rate_density,
)


roots = [0.48952076-9.02994296e-09j 0.48952076+9.02994296e-09j]
n_alpha = (3.4266452895611744e+19-632096007319.7186j)


(Very slightly) complex roots.

In [34]:
# Try more favourable values (found at LT, ne = 9e19)
proton_rate_density = 7.70227336206380e14
fusden_plasma_alpha = 3.0937573966150234e17
t_alpha_confinement = 14.700927832497008
znimp = 3.0994787703099156e18
ne = 9e19

# Modification to rates
fusden_plasma_alpha = fusden_plasma_alpha / ne**2
proton_rate_density = proton_rate_density / ne**2

calc_n_alpha(
    ne=ne,
    t_alpha_confinement=t_alpha_confinement,
    fusden_plasma_alpha=fusden_plasma_alpha,
    nd_beam_ions=nd_beam_ions,
    znimp=znimp,
    z_he=z_he,
    proton_rate_density=proton_rate_density,
)

roots = [0.03967882 5.85949807]
n_alpha = 3.5710940805129257e+18


So, can get real roots using values from LT evaluation at higher ne. This is a very high n_alpha: it's half of n_e, compared to 8.2e18 in original LT soln case (~5x less). It's almost a repeated root. As the roots are a fraction of ne (the concentration), discard root > 1 if required, although this doesn't seem to be the case for these values. After some wiggling of values, roots don't appear to be very sensitive to any one value in particular.

## Put into Process: single evaluation
Try single evaluation run to get quadratic solution for nd_alphas. 

In [28]:
INPUT_FILE = "data/lt_tau_alpha_sol_eval_IN.DAT"


def eval_cons(ne, te):
    # Set up
    single_run = SingleRun(INPUT_FILE)
    process.data_structure.impurity_radiation_module.f_nd_impurity_electron_array[
        13
    ] = 5.0e-6
    process.data_structure.current_drive_variables.p_hcd_primary_extra_heat_mw = 200.0
    impurity_radiation.rho_fix = True
    process.data_structure.impurity_radiation_module.f_p_plasma_core_rad_reduction = 1.0
    impurity_radiation.int_edge_rad = True
    process.data_structure.physics_variables.i_rad_loss = 0
    process.physics.old_n_alpha_calc = False
    process.physics.current_n_alpha_calc = False
    process.physics.eval_count = 0
    process.physics.mattis_test = False

    # Modifications for each run
    process.data_structure.physics_variables.dene = ne
    process.data_structure.physics_variables.te = te

    single_run.run()
    n_alpha = process.data_structure.physics_variables.nd_alphas
    return n_alpha


ne = 9.0e19
te = 9.0
ne_grid, te_grid = np.meshgrid(ne, te)
eval_cons_vec = np.vectorize(pyfunc=eval_cons)
n_alpha = eval_cons_vec(ne_grid, te_grid)
clear_output()
print(n_alpha)

dr_tf_plasma_case to small to accommodate the WP, forced to minimum value
Ratio of central solenoid overall current density at beginning of flat-top / end of flat-top > 1 (|f_j_cs_start_end_flat_top| > 1)
dr_tf_plasma_case to small to accommodate the WP, forced to minimum value


The IN.DAT file does not contain any obsolete variables.
 
**************************************************************************************************************
************************************************** PROCESS ***************************************************
************************************** Power Reactor Optimisation Code ***************************************
**************************************************************************************************************
 
Version : 3.2.0
Git Tag : 
Git Branch : 
Date : 17/10/2025 UTC
Time : 11:19
User : jon
Computer : jon-Precision-3560
Directory : /home/jon/code/notebooks/solutions_to_process/solutions_to_process_3
Input : /home/jon/code/notebooks/solutions_to_process/solutions_to_process_3/data/lt_tau_alpha_sol_eval_IN.DAT
Run title : generic large tokamak
Run type : Reactor concept design: Pulsed tokamak model, (c) UK Atomic Energy Authority
 
**************************************************************

ValueError: 2 physical roots for c_alpha: [0.48259648-7.32206252e-09j 0.48259648+7.32206252e-09j]

Errors due to 2 roots (slightly complex). Produces a very high c_alpha. Try with Matti's values from an Apollo test. This requires hardcoding Matti's test values in with a switch.

In [29]:
INPUT_FILE = "data/lt_tau_alpha_sol_eval_IN.DAT"


def eval_cons(ne, te):
    # Set up
    single_run = SingleRun(INPUT_FILE)
    # Xe
    process.data_structure.impurity_radiation_module.f_nd_impurity_electron_array[
        12
    ] = 3.8168e-4
    # W
    process.data_structure.impurity_radiation_module.f_nd_impurity_electron_array[
        13
    ] = 5.0e-5
    process.data_structure.current_drive_variables.p_hcd_primary_extra_heat_mw = 200.0
    impurity_radiation.rho_fix = True
    process.data_structure.impurity_radiation_module.f_p_plasma_core_rad_reduction = 1.0
    impurity_radiation.int_edge_rad = True
    process.data_structure.physics_variables.i_rad_loss = 0
    process.physics.old_n_alpha_calc = False
    process.physics.current_n_alpha_calc = False
    process.physics.eval_count = 0

    # Matti's test case
    process.physics.mattis_test = True

    # Modifications for each run
    process.data_structure.physics_variables.dene = ne
    process.data_structure.physics_variables.te = te

    single_run.run()
    n_alpha = process.data_structure.physics_variables.nd_alphas
    return n_alpha


ne = 6.8711e19
te = 1.0926e1
ne_grid, te_grid = np.meshgrid(ne, te)
eval_cons_vec = np.vectorize(pyfunc=eval_cons)
n_alpha = eval_cons_vec(ne_grid, te_grid)
clear_output()
print(n_alpha)

dr_tf_plasma_case to small to accommodate the WP, forced to minimum value
Ratio of central solenoid overall current density at beginning of flat-top / end of flat-top > 1 (|f_j_cs_start_end_flat_top| > 1)
dr_tf_plasma_case to small to accommodate the WP, forced to minimum value


The IN.DAT file does not contain any obsolete variables.
 
**************************************************************************************************************
************************************************** PROCESS ***************************************************
************************************** Power Reactor Optimisation Code ***************************************
**************************************************************************************************************
 
Version : 3.2.0
Git Tag : 
Git Branch : 
Date : 17/10/2025 UTC
Time : 11:20
User : jon
Computer : jon-Precision-3560
Directory : /home/jon/code/notebooks/solutions_to_process/solutions_to_process_3
Input : /home/jon/code/notebooks/solutions_to_process/solutions_to_process_3/data/lt_tau_alpha_sol_eval_IN.DAT
Run title : generic large tokamak
Run type : Reactor concept design: Pulsed tokamak model, (c) UK Atomic Energy Authority
 
**************************************************************

ValueError: 2 physical roots for c_alpha: [0.48884512 0.48884513]

Matti's test case appears to result in 2 (almost repeated) real roots. The c_alpha roots are still very high, and very similar to the real parts of the complex roots in the previous case.